# Notebook 04: Full End-to-End Fine-Tuning

In this notebook, we perform true end-to-end training. **All weights of the Moirai encoder and the classification heads are updated**.


In [1]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import copy
from IPython.display import display

from tslearn.datasets import UCR_UEA_datasets
from sklearn.preprocessing import LabelEncoder
from uni2ts.model.moirai import MoiraiModule
from encoder import MoiraiEncoder

from heads import (
    MeanPoolingClassifier, 
    SingleScaleAttentionClassifier, 
    SingleScaleMultiHeadClassifier, 
    HierarchicalMultiHeadClassifier
)
from models import MoiraiClassifier, MoiraiMaskTuner
from utils import preprocess_data, create_raw_dataloaders, train_finetune

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_VARS = 6
SIZE = "small"
PATCH_SIZES = [16] #[8, 16, 32, 64]
BATCH_SIZE = 32 

/home/onyxia/work/DLforTimeSeries/moirai-classification/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def grid_search_full_finetune(model_class, model_kwargs, train_loader, val_loader, test_loader, device="cuda"):
    lr_grid = [5e-5, 1e-5]
    wd_grid = [0.01, 0.05]
    best_val_loss = float('inf')
    best_model = None
    
    for wd in wd_grid:
        for lr in lr_grid:
            model = model_class(**model_kwargs).to(device)
            val_loss, trained_model = train_finetune(
                model=model, train_loader=train_loader, val_loader=val_loader,
                lr=lr, epochs=500, weight_decay=wd, device=device
            )
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_model = copy.deepcopy(trained_model)     
            
    best_model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for b_t, b_o, b_p, b_y in test_loader:
            b_t, b_o, b_p, b_y = b_t.to(device), b_o.to(device), b_p.to(device), b_y.to(device)
            preds = torch.argmax(best_model(b_t, b_o, b_p), dim=-1)
            correct += (preds == b_y).sum().item()
            total += b_y.size(0)
            
    return best_model, correct / total

class FullMaskOnlyWrapper(nn.Module):
    def __init__(self, patch_size, num_vars, num_classes, size="small"):
        super().__init__()
        moirai_enc = MoiraiEncoder(
            module=MoiraiModule.from_pretrained(f"Salesforce/moirai-1.1-R-{size}"),
            prediction_length=patch_size, context_length=36, patch_size=patch_size, 
            num_samples=100, target_dim=num_vars, feat_dynamic_real_dim=0, past_feat_dynamic_real_dim=0,
        )
        # No freezing 
        self.model = MoiraiMaskTuner(encoder=moirai_enc, num_vars=num_vars, num_classes=num_classes)

    def forward(self, t, o, p):
        return self.model(t, o, p)

class FullHeadWrapper(nn.Module):
    def __init__(self, head_class, head_kwargs, patch_size, num_vars, size="small", remove_last_patch=False):
        super().__init__()
        moirai_enc = MoiraiEncoder(
            module=MoiraiModule.from_pretrained(f"Salesforce/moirai-1.1-R-{size}"),
            prediction_length=patch_size, context_length=36, patch_size=patch_size, 
            num_samples=100, target_dim=num_vars, feat_dynamic_real_dim=0, past_feat_dynamic_real_dim=0,
        )
        # No freezing!
        head = head_class(**head_kwargs)
        self.model = MoiraiClassifier(encoder=moirai_enc, head=head, remove_last_patch=remove_last_patch, num_vars=num_vars)

    def forward(self, t, o, p):
        return self.model(t, o, p)

In [3]:
ds = UCR_UEA_datasets()
X_train, y_train, X_test, y_test = ds.load_dataset("LSST")

le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)
num_classes = len(set(y_train_encoded))

X_tr_t, X_tr_o, X_tr_p = preprocess_data(X_train, device=DEVICE)
X_te_t, X_te_o, X_te_p = preprocess_data(X_test, device=DEVICE)

y_tr_t = torch.tensor(y_train_encoded, dtype=torch.long, device=DEVICE)
y_te_t = torch.tensor(y_test_encoded, dtype=torch.long, device=DEVICE)

tr_loader, va_loader = create_raw_dataloaders(X_tr_t, X_tr_o, X_tr_p, y_tr_t, batch_size=BATCH_SIZE, device=DEVICE)
te_loader = torch.utils.data.DataLoader(
    torch.utils.data.TensorDataset(X_te_t, X_te_o, X_te_p, y_te_t), batch_size=BATCH_SIZE, shuffle=False
)

## 1. Baseline: Linear Model on Mask Embedding Only (Full FT)

In [ ]:
df_mask_only = pd.DataFrame(index=["Mask Only"], columns=PATCH_SIZES)
df_mask_only.columns.name = "Patch Size"

for patch in PATCH_SIZES:
    _, test_acc = grid_search_full_finetune(
        model_class=FullMaskOnlyWrapper,
        model_kwargs={"patch_size": patch, "num_vars": NUM_VARS, "num_classes": num_classes, "size": SIZE},
        train_loader=tr_loader, val_loader=va_loader, test_loader=te_loader, device=DEVICE
    )
    df_mask_only.loc["Mask Only", patch] = test_acc

display(df_mask_only.astype(float).round(4))

## 2. Baseline: Mean Pooling

In [ ]:
df_mean_pool = pd.DataFrame(index=["Mean Pooling"], columns=PATCH_SIZES)
df_mean_pool.columns.name = "Patch Size"

for patch in PATCH_SIZES:
    _, test_acc = grid_search_full_finetune(
        model_class=FullHeadWrapper,
        model_kwargs={
            "head_class": MeanPoolingClassifier,
            "head_kwargs": {"num_vars": NUM_VARS, "num_classes": num_classes},
            "patch_size": patch, "num_vars": NUM_VARS, "size": SIZE, "remove_last_patch": False
        },
        train_loader=tr_loader, val_loader=va_loader, test_loader=te_loader, device=DEVICE
    )
    df_mean_pool.loc["Mean Pooling", patch] = test_acc

display(df_mean_pool.astype(float).round(4))

## 3. Advanced Pooling: Attention & MHA (Full FT)

In [ ]:
PATCH_SIZES_TO_TEST = [8, 16] 
MODES = ["shared_context", "independent_context"]
NUM_HEADS = 16

model_names_single = ["Attention (Base)", f"MHA (Heads={NUM_HEADS})"]
index_single = pd.MultiIndex.from_product([model_names_single, MODES], names=["Model", "Mode"])
df_adv_single = pd.DataFrame(index=index_single, columns=PATCH_SIZES_TO_TEST)
df_adv_single.columns.name = "Patch Size"

for patch in PATCH_SIZES_TO_TEST:
    for mode in MODES:
        # 1. Basic Attention
        _, acc_attn = grid_search_full_finetune(
            model_class=FullHeadWrapper,
            model_kwargs={
                "head_class": SingleScaleAttentionClassifier,
                "head_kwargs": {"num_vars": NUM_VARS, "num_classes": num_classes, "mode": mode},
                "patch_size": patch, "num_vars": NUM_VARS, "size": SIZE, "remove_last_patch": False
            },
            train_loader=tr_loader, val_loader=va_loader, test_loader=te_loader, device=DEVICE
        )
        df_adv_single.loc[(model_names_single[0], mode), patch] = acc_attn

        # 2. Multi-Head Attention
        _, acc_mha = grid_search_full_finetune(
            model_class=FullHeadWrapper,
            model_kwargs={
                "head_class": SingleScaleMultiHeadClassifier,
                "head_kwargs": {"num_vars": NUM_VARS, "num_classes": num_classes, "patch_mode": mode, "num_heads": NUM_HEADS},
                "patch_size": patch, "num_vars": NUM_VARS, "size": SIZE, "remove_last_patch": False
            },
            train_loader=tr_loader, val_loader=va_loader, test_loader=te_loader, device=DEVICE
        )
        df_adv_single.loc[(model_names_single[1], mode), patch] = acc_mha

display(df_adv_single.astype(float).round(4))

## 4. Hierarchical MHA (Full FT)

In [ ]:
df_hierarchical = pd.DataFrame(index=MODES, columns=PATCH_SIZES_TO_TEST)
df_hierarchical.index.name = "Mode"
df_hierarchical.columns.name = "Patch Size"

for patch in PATCH_SIZES_TO_TEST:
    for mode in MODES:
        _, acc_hier = grid_search_full_finetune(
            model_class=FullHeadWrapper,
            model_kwargs={
                "head_class": HierarchicalMultiHeadClassifier,
                "head_kwargs": {"num_vars": NUM_VARS, "num_classes": num_classes, "patch_mode": mode, "num_heads": NUM_HEADS},
                "patch_size": patch, "num_vars": NUM_VARS, "size": SIZE, "remove_last_patch": False
            },
            train_loader=tr_loader, val_loader=va_loader, test_loader=te_loader, device=DEVICE
        )
        df_hierarchical.loc[mode, patch] = acc_hier

display(df_hierarchical.astype(float).round(4))

## 5. Ablation Study: Number of Attention Heads

In [ ]:
PATCH = 16
HEADS_TO_TEST = [16, 32, 64, 128] 

df_heads_ablation = pd.DataFrame(index=MODES, columns=HEADS_TO_TEST)
df_heads_ablation.index.name = "Mode"
df_heads_ablation.columns.name = f"Num Heads (Patch {PATCH})"

for mode in MODES:
    for heads in HEADS_TO_TEST:
        _, test_acc = grid_search_full_finetune(
            model_class=FullHeadWrapper,
            model_kwargs={
                "head_class": SingleScaleMultiHeadClassifier,
                "head_kwargs": {"num_vars": NUM_VARS, "num_classes": num_classes, "patch_mode": mode, "num_heads": heads},
                "patch_size": PATCH, "num_vars": NUM_VARS, "size": SIZE, "remove_last_patch": False
            },
            train_loader=tr_loader, val_loader=va_loader, test_loader=te_loader, device=DEVICE
        )
        df_heads_ablation.loc[mode, heads] = test_acc

display(df_heads_ablation.astype(float).round(4))